# Electivo de Bioinformática — Clase 6

## Un script para QC, alineamiento y mejoramiento de BAM

**Programa:** Doctorado — 2º año
**Duración:** 3 horas
**Fecha:** 29 de septiembre

### Objetivos de la clase

Al finalizar esta clase, serán capaces de:

1. Recordar qué responde cada una de las tres etapas de este flujo: **control de calidad** (FastQC), **alineamiento** (`bwa`) y **mejoramiento del BAM** (sincronizar parejas, marcar duplicados y recalibrar calidades).
2. Verificar que el ambiente y los datos de la clase anterior estén listos para seguir trabajando, sin reinstalar ni regenerar nada de cero.
3. Escribir, paso a paso y **con nano**, un script bash que encadene las cuatro etapas para una muestra, con parámetros y comprobaciones básicas.
4. Ejecutar el script sobre una muestra nueva y verificar el resultado.

Hoy **no instalamos nada nuevo ni preparamos datos**: eso ya quedó hecho en la clase 5. El foco de hoy es **escribir el script** que encadena lo que la clase pasada hicimos herramienta por herramienta.

---
## 1. Repaso: las etapas del flujo (5 min)

| Etapa | ¿Qué problema resuelve? | Herramienta |
|---|---|---|
| Control de calidad | ¿Los datos crudos son de buena calidad? | `fastqc` |
| Alineamiento | ¿De qué posición del genoma viene cada lectura? | `bwa mem` + `samtools sort` |
| Sincronizar parejas | Que cada lectura y su pareja coincidan en dónde dice cada una que está la otra | `picard FixMateInformation` |
| Marcar duplicados | Que una misma molécula, copiada varias veces por PCR, no se cuente como varias lecturas independientes | `picard MarkDuplicates` |
| Recalibrar calidades | Corregir el sesgo de las calidades que reporta el secuenciador, comparando con variantes conocidas | `gatk BaseRecalibrator` + `ApplyBQSR` |

Las cinco herramientas ya las usamos por separado en la clase 5. Hoy las encadenamos en **un solo script**.

---
## 2. Verificar que el ambiente esté listo (15 min)

Todo lo que necesitamos —el ambiente `mejoramiento`, la referencia indexada, el FASTQ y los sitios conocidos— ya quedó creado en la clase 5, en `~/bioinfo/clase5/`. No lo vamos a rehacer: solo lo **comprobamos**, igual que hace un script antes de correr.

Si guardaron esos archivos en otra carpeta, ajusten la variable `CLASE5` de la celda siguiente antes de correrla.

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

CLASE5=~/bioinfo/clase5

echo "=== Herramientas en el ambiente 'mejoramiento' ==="
for HERR in fastqc bwa samtools picard gatk; do
    if command -v "$HERR" >/dev/null 2>&1; then
        echo "OK    $HERR"
    else
        echo "FALTA $HERR"
    fi
done

echo
echo "=== Referencia e indices ($CLASE5/referencia) ==="
REF="$CLASE5/referencia/chrM.fasta"
for ARCHIVO in "$REF" "${REF}.fai" "${REF}.bwt" "${REF%.*}.dict"; do
    if [ -s "$ARCHIVO" ]; then echo "OK    $ARCHIVO"; else echo "FALTA $ARCHIVO"; fi
done

echo
echo "=== FASTQ y sitios conocidos ($CLASE5) ==="
for ARCHIVO in "$CLASE5/fastq/mtDNA_sample_R1.fastq.gz" \
               "$CLASE5/fastq/mtDNA_sample_R2.fastq.gz" \
               "$CLASE5/sitios_conocidos.vcf"; do
    if [ -s "$ARCHIVO" ]; then echo "OK    $ARCHIVO"; else echo "FALTA $ARCHIVO"; fi
done

Si algo aparece como **FALTA**:

- Una **herramienta**: revisen que activaron el ambiente correcto, o agréguenla con `conda install -n mejoramiento -c bioconda <nombre>` (por ejemplo, `fastqc` puede faltar en `mejoramiento` si en la clase 5 solo se instaló para `bwa`/`samtools`/`picard`/`gatk4`: en ese caso, `conda install -n mejoramiento -c bioconda fastqc` alcanza, no hace falta recrear el ambiente completo).
- Un **archivo**: revisen la sección 3 de la clase 5. Si de verdad no está en ningún lado, al final de este notebook hay un apéndice con los comandos mínimos para regenerarlo.

Si todo salió **OK**, seguimos.

---
## 3. Repaso rápido: parámetros, comprobaciones y nano (15 min)

El script de hoy va a recibir varios datos como **parámetros posicionales** (`$1`, `$2`, `$3`...) y va a **comprobar** que existan antes de usarlos.

Hasta ahora, en el notebook armábamos los scripts con `cat > archivo << 'EOF' ...`. Eso funciona, pero no es como ustedes van a escribir scripts de verdad: en el servidor lo hacemos con **`nano`**, y en el computador personal —si prefieren— con **Sublime Text**, subiendo después el archivo al servidor (con `scp` o `rsync`, como vimos en la clase 1). Desde ahora, cuando el notebook les muestre el contenido de un script, ustedes lo escriben en su editor, no en la celda.

Un ejemplo chico para refrescar la mecánica de `nano`:

1. En la terminal del servidor: `cd ~/bioinfo/clase6` (creen la carpeta si no existe: `mkdir -p ~/bioinfo/clase6`).
2. `nano saludo.sh`
3. Peguen (o escriban) exactamente esto:

```bash
#!/bin/bash
NOMBRE="$1"
ARCHIVO="$2"

if [ -z "$NOMBRE" ] || [ -z "$ARCHIVO" ]; then
    echo "Uso: $0 <nombre> <archivo>"
    exit 1
fi

if [ ! -f "$ARCHIVO" ]; then
    echo "ERROR: no encuentro el archivo '$ARCHIVO'"
    exit 1
fi

echo "Hola $NOMBRE, el archivo '$ARCHIVO' existe."
```

4. Guarden con `Ctrl+O` y `Enter` (confirma el nombre del archivo), y salgan con `Ctrl+X`.
5. De vuelta en la terminal: `chmod +x saludo.sh`.

Ahora sí, esto lo corremos en una celda (correr comandos en la terminal, copiando y pegando, es exactamente lo que hacíamos hasta ahora):

In [ ]:
%%bash
cd ~/bioinfo/clase6

echo "--- Con todo correcto ---"
./saludo.sh Ana ~/bioinfo/clase5/referencia/chrM.fasta

echo
echo "--- Con un archivo que no existe ---"
./saludo.sh Ana archivo_inventado.fasta; echo "codigo de salida: $?"

Recuerden comillas alrededor de las variables (`"$ARCHIVO"`, no `$ARCHIVO`): si un nombre tuviera un espacio, sin comillas el script se rompe.

---
## 4. Construyendo el script, en nano (70 min)

Vamos a escribir `mejorar_bam.sh` **por partes**, con `nano`, en `~/bioinfo/clase6/`. El script recibe **seis** parámetros: la muestra, la referencia, los dos FASTQ, el archivo de sitios conocidos y una carpeta de salida.

Para cada parte: la pegan en `nano` donde se indica, guardan (`Ctrl+O`, `Enter`) y siguen editando (no hace falta salir y volver a entrar entre partes, pero si lo hacen, el atajo `Alt+/` en `nano` los deja al final del archivo, listos para agregar la siguiente parte; `Alt+\` los deja al principio).

### 4.1 Encabezado, parámetros y comprobaciones

`cd ~/bioinfo/clase6` y `nano mejorar_bam.sh` (archivo nuevo). Escriban:

```bash
#!/bin/bash
#
# mejorar_bam.sh
# QC + alineamiento + mejoramiento del BAM (parejas, duplicados, calidades).
#
# Uso: ./mejorar_bam.sh <muestra> <referencia.fasta> <R1.fastq.gz> <R2.fastq.gz> <sitios_conocidos.vcf> <directorio_salida>
#
# Asume que la referencia ya esta indexada (bwa index, samtools faidx, .dict).

set -e   # si un comando falla, el script se detiene aqui

# ---------- Parametros ----------
MUESTRA="$1"
REF="$2"
R1="$3"
R2="$4"
SITIOS="$5"
SALIDA="$6"

# ---------- Comprobaciones ----------
if [ -z "$SALIDA" ]; then
    echo "Uso: $0 <muestra> <referencia.fasta> <R1.fastq.gz> <R2.fastq.gz> <sitios_conocidos.vcf> <directorio_salida>"
    exit 1
fi

for ARCHIVO in "$REF" "$R1" "$R2" "$SITIOS"; do
    if [ ! -f "$ARCHIVO" ]; then
        echo "ERROR: no encuentro el archivo '$ARCHIVO'"
        exit 1
    fi
done

mkdir -p "$SALIDA/fastqc" "$SALIDA/alineamiento"
echo "=== Muestra: $MUESTRA ==="
```

Guarden y sigan editando.

### 4.2 Paso 1: control de calidad

Al final del archivo, agreguen:

```bash

# ---------- Paso 1: control de calidad ----------
echo "--- [1/4] FastQC ---"
fastqc -o "$SALIDA/fastqc" "$R1" "$R2"
```

### 4.3 Paso 2: alineamiento

Al final del archivo, agreguen:

```bash

# ---------- Paso 2: alineamiento ----------
echo "--- [2/4] bwa mem + samtools sort ---"
BAM_ORDENADO="$SALIDA/alineamiento/${MUESTRA}.sorted.bam"

bwa mem -R "@RG\tID:${MUESTRA}\tSM:${MUESTRA}\tLB:${MUESTRA}\tPL:ILLUMINA" \
    "$REF" "$R1" "$R2" \
  | samtools sort -o "$BAM_ORDENADO" -

samtools index "$BAM_ORDENADO"
```

### 4.4 Paso 3: sincronizar parejas y marcar duplicados

Dos comandos de Picard, uno después del otro: `FixMateInformation` primero, `MarkDuplicates` después (que también genera un archivo de métricas). Al final del archivo, agreguen:

```bash

# ---------- Paso 3: sincronizar parejas y marcar duplicados ----------
echo "--- [3/4] Picard FixMateInformation + MarkDuplicates ---"
BAM_FIXMATE="$SALIDA/alineamiento/${MUESTRA}.fixmate.bam"
BAM_MARKDUP="$SALIDA/alineamiento/${MUESTRA}.markdup.bam"

picard FixMateInformation -I "$BAM_ORDENADO" -O "$BAM_FIXMATE"

picard MarkDuplicates \
    -I "$BAM_FIXMATE" -O "$BAM_MARKDUP" \
    -M "$SALIDA/${MUESTRA}.dup_metrics.txt"

samtools index "$BAM_MARKDUP"
```

### 4.5 Paso 4: recalibrar las calidades (BQSR)

Dos comandos de GATK: `BaseRecalibrator` calcula **cómo** hay que corregir las calidades (y guarda el resultado en una tabla), `ApplyBQSR` las corrige de verdad. Al final del archivo, agreguen:

```bash

# ---------- Paso 4: recalibrar calidades (BQSR) ----------
echo "--- [4/4] GATK BaseRecalibrator + ApplyBQSR ---"
TABLA="$SALIDA/${MUESTRA}.recal_data.table"
BAM_FINAL="$SALIDA/${MUESTRA}.final.bam"

gatk IndexFeatureFile -I "$SITIOS" > /dev/null

gatk BaseRecalibrator \
    -I "$BAM_MARKDUP" -R "$REF" --known-sites "$SITIOS" \
    -O "$TABLA" > /dev/null

gatk ApplyBQSR \
    -I "$BAM_MARKDUP" -R "$REF" --bqsr-recal-file "$TABLA" \
    -O "$BAM_FINAL" > /dev/null

echo "--- Listo: $BAM_FINAL ---"
samtools flagstat "$BAM_FINAL" > "$SALIDA/${MUESTRA}.flagstat.txt"
cat "$SALIDA/${MUESTRA}.flagstat.txt"
```

Guarden y salgan de `nano` (`Ctrl+O`, `Enter`, `Ctrl+X`). El script ya está completo.

### 4.6 Revisar y comprobar la sintaxis

De vuelta en la terminal (esto sí lo corremos en una celda, porque son comandos que se ejecutan, no contenido para escribir en un archivo):

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase6
chmod +x mejorar_bam.sh
cat -n mejorar_bam.sh

`bash -n` revisa que el script esté bien formado (comillas, `fi`/`done` que falten) **sin ejecutarlo** — conviene hacerlo antes de correr algo que tarda varios minutos. Si el resultado de la celda anterior no coincide con lo que muestra este notebook, revísenlo en `nano` antes de seguir.

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase6
bash -n mejorar_bam.sh && echo "sintaxis OK"

---
## 5. Ejecutarlo y revisar el resultado (20 min)

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase6
mkdir -p resultados

./mejorar_bam.sh mtDNA_sample ~/bioinfo/clase5/referencia/chrM.fasta \
    ~/bioinfo/clase5/fastq/mtDNA_sample_R1.fastq.gz ~/bioinfo/clase5/fastq/mtDNA_sample_R2.fastq.gz \
    ~/bioinfo/clase5/sitios_conocidos.vcf resultados

Revisemos qué dejó el script:

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase6/resultados

echo "--- Contenido de resultados/ ---"
ls -la

echo
echo "--- Duplicados detectados (Picard) ---"
grep -A1 '^LIBRARY' mtDNA_sample.dup_metrics.txt | cut -f1,3,7-9

echo
echo "--- Calidad promedio: antes y despues de BQSR ---"
echo "antes:   $(samtools stats alineamiento/mtDNA_sample.markdup.bam | grep -m1 'average quality')"
echo "despues: $(samtools stats mtDNA_sample.final.bam | grep -m1 'average quality')"

El BAM final (`resultados/mtDNA_sample.final.bam`) es el insumo directo para el llamado de variantes: ordenado, indexado, con las parejas sincronizadas, los duplicados marcados y las calidades recalibradas.

> **Para discutir (en parejas, 3 min):** (a) ¿Qué pasa si ejecutan el script una segunda vez con la misma muestra? (b) ¿Por qué `MarkDuplicates` necesita un archivo de métricas (`-M`) además del BAM de salida, y `FixMateInformation` no?

---
## 6. Ejercicio integrador (en parejas, 20 min)

1. **Generen una muestra nueva** con `dwgsim`, usando una semilla (`-z`) distinta a la del ejemplo (por ejemplo, los últimos dígitos del RUT de alguno de los dos) y otro nombre (por ejemplo `mtDNA_pareja`). Mismos parámetros que en la clase 5, cambiando solo `-z` y el nombre de salida; guárdenla en `~/bioinfo/clase6/fastq/` (creen la carpeta si no existe) y renombren a `_R1.fastq.gz`/`_R2.fastq.gz`.
2. **Copien y renombren** el archivo de mutaciones de su muestra como sitios conocidos (`cd ~/bioinfo/clase6 && cp fastq/mtDNA_pareja.mutations.vcf sitios_conocidos_pareja.vcf`, o el nombre que corresponda).
3. **Corran el script** sobre esa muestra nueva, sin modificarlo, usando la referencia de `~/bioinfo/clase5/` (esa no cambia).
4. **Reporten:** ¿cuántas lecturas totales tiene su BAM final? ¿Cuántas quedaron marcadas como duplicado? ¿La calidad promedio subió, bajó o se mantuvo igual después de BQSR?
5. **Desafío (si les alcanza el tiempo):** agreguen al script, con `nano`, una comprobación de que `$MUESTRA` no esté vacío (hoy solo se comprueba `$SALIDA`). Guarden la versión modificada como `mejorar_bam_v2.sh` (cópienlo primero, no lo editen encima del original: `cp mejorar_bam.sh mejorar_bam_v2.sh` y editen la copia).

No hay una única forma correcta de resolverlo. Lo importante es que puedan explicar qué hace cada parámetro que usaron.

---
## 7. Cierre y resumen

### Comandos vistos hoy

```
Verificacion:  command -v <herramienta> ; [ -s archivo ]
QC:            fastqc -o carpeta R1.fastq.gz R2.fastq.gz
Alineamiento:  bwa mem -R '@RG...' ref.fasta R1.fastq.gz R2.fastq.gz | samtools sort -o salida.bam -
Parejas:       picard FixMateInformation -I in.bam -O out.bam
Duplicados:    picard MarkDuplicates -I in.bam -O out.bam -M metricas.txt
Calidades:     gatk BaseRecalibrator -I in.bam -R ref.fasta --known-sites sitios.vcf -O tabla.table
               gatk ApplyBQSR -I in.bam -R ref.fasta --bqsr-recal-file tabla.table -O out.bam
Script:        parametros posicionales ($1, $2, ...), comprobaciones (-z, -f), set -e, comillas en variables
Editor:        nano archivo.sh   (Ctrl+O guarda, Ctrl+X sale, Alt+/ va al final, Alt+\ va al principio)
```

### Ideas clave

- El mejoramiento de un BAM tiene tres pasos cortos, siempre en el mismo orden: **sincronizar parejas → marcar duplicados → recalibrar calidades**.
- Un script con **parámetros** (en vez de nombres fijos) se puede reusar con cualquier muestra sin tocar su contenido.
- **Comprobar antes de empezar** (¿existen los archivos? ¿vinieron todos los parámetros? ¿está instalado lo que necesito?) evita que el script —o la clase— avance con algo incompleto y falle más adelante con un error confuso.

### Lo que dejamos listo para la próxima clase

- El script `~/bioinfo/clase6/mejorar_bam.sh`, probado con al menos dos muestras.
- Un BAM final por muestra (`resultados/*.final.bam`, con su índice): el insumo directo para el llamado de variantes.

### Tarea para antes de la próxima clase

1. Verifiquen que `resultados/mtDNA_sample.final.bam` y el de su pareja siguen disponibles (con sus `.bai`).
2. Terminen el ejercicio integrador si no alcanzaron.
3. Repasen brevemente los nombres de los tres pasos del mejoramiento y qué herramienta hace cada uno (sección 1): la próxima clase los vamos a dar por conocidos.

---
## Apéndice: si algo faltó en la sección 2

Solo corran lo que efectivamente les hizo falta, no todo el bloque.

```bash
# Agregar una sola herramienta faltante al ambiente mejoramiento
conda install -n mejoramiento -c bioconda <nombre_herramienta>

# Recrear el ambiente completo desde cero (si no existe)
conda create -y -n mejoramiento -c conda-forge -c bioconda bwa samtools fastqc picard gatk4 dwgsim

# Reindexar la referencia (si falta algun indice)
cd ~/bioinfo/clase5/referencia
bwa index chrM.fasta
samtools faidx chrM.fasta
picard CreateSequenceDictionary -R chrM.fasta -O chrM.dict

# Regenerar el FASTQ pareado (si no existe en ninguna clase anterior)
cd ~/bioinfo/clase5/fastq
dwgsim -N 2000 -1 100 -2 100 -e 0.001 -E 0.001 -r 0.0006 -R 0 -X 0 -y 0 -z 7 -c 0 \
    ../referencia/chrM.fasta mtDNA_sample
mv mtDNA_sample.bwa.read1.fastq.gz mtDNA_sample_R1.fastq.gz
mv mtDNA_sample.bwa.read2.fastq.gz mtDNA_sample_R2.fastq.gz

# Sitios conocidos (a partir del VCF que genera dwgsim)
cd ~/bioinfo/clase5
cp fastq/mtDNA_sample.mutations.vcf sitios_conocidos.vcf
```